<a href="https://colab.research.google.com/github/OnzyBoy/Financial-Inclusion-in-East-Africa/blob/main/ZindiFinancialInclusion.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Financial Inclusion in Africa

**Problem Statement:**

> Despite the rise of mobile money in East Africa, a significant portion of the population remains formally "unbanked." Financial institutions lack a data-driven way to predict which individuals are likely to adopt formal banking services based on their socioeconomic profiles. This project aims to build a predictive model to identify potential customers for financial inclusion, allowing stakeholders to target resources more efficiently and design products that bridge the digital divide.

Dataset Link: [Zindi Financial Inclusion in Africa](https://zindi.africa/competitions/financial-inclusion-in-africa/data)



## Downloads & Imports

In [ ]:
!pip install ydata-profiling

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from ydata_profiling import ProfileReport

## Data Loading & Inspection

In [ ]:
df_description = pd.read_csv('/content/VariableDefinitions.csv')
df_description.head(14)

In [ ]:
df = pd.read_csv('/content/Train.csv')
df.head(7)

In [ ]:
df.info()

In [ ]:
df.shape

In [ ]:
df.describe(include='all')

In [ ]:
df.duplicated().sum() # No duplicated data

In [ ]:
df.isna().sum() # No missing data

In [ ]:
profile = ProfileReport(df, title="Profiling Report", explorative = True)
profile.to_notebook_iframe()

Dataset is clean, no missing values, no duplicated data.

Target Variable: `bank_account` - To determine what variables lead to a person having a bank account or not.

Correlations (Pandas Profiling):
- `country` is highly correlated with `year`.
- `gender_of_respondent` is highly correlated with `relationship_with_head`.
- `relationship_with_head` is highly correlated with `gender_of_respondent`.

Key Observations:

- Target Imbalance: Out of 23,524 people, 20,212 said "No". That means only ~14% of your dataset has a bank account.

Why this matters: If your model just guesses "No" every time, it will be 86% accurate but totally useless. We will need to focus on F1-Score or Balanced Accuracy later.

- The "Age" Outlier: Your max age is 100, while the 75% mark is 49.

Why this matters: You have some very senior respondents. While not necessarily "wrong," we should check if these are outliers or if they represent a specific demographic (e.g., retirees with pensions/bank accounts).

- The "Household Size" Outlier: You have a max household size of 21.

Why this matters: In a Kenyan/East African context, large extended families are real, but 21 is a massive jump from the mean of 3.7. We should visualize this to see if it's a data entry error or just a very large homestead.



In [ ]:
df.columns

In [ ]:
df['bank_account'].value_counts()

### Outlier Checks

In [ ]:
# Create boxplot for household_size
plt.figure(figsize=(8, 6))
sns.boxplot(y=df['household_size'])
plt.title('Boxplot of Household Size')
plt.ylabel('Household Size')
plt.show()

# Create boxplot for age_of_respondent
plt.figure(figsize=(8, 6))
sns.boxplot(y=df['age_of_respondent'])
plt.title('Boxplot of Age of Respondent')
plt.ylabel('Age of Respondent')
plt.show()

In [ ]:
def count_outliers_iqr(df, column):
    Q1 = df[column].quantile(0.25)
    Q3 = df[column].quantile(0.75)
    IQR = Q3 - Q1
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR

    outliers = df[(df[column] < lower_bound) | (df[column] > upper_bound)]
    return len(outliers)

# Count outliers for 'household_size'
outliers_household_size = count_outliers_iqr(df, 'household_size')
print(f"Number of outliers in 'household_size': {outliers_household_size}")

# Count outliers for 'age_of_respondent'
outliers_age_of_respondent = count_outliers_iqr(df, 'age_of_respondent')
print(f"Number of outliers in 'age_of_respondent': {outliers_age_of_respondent}")

Option A: Capping (Winsorization): You set a limit. For example, any household size > 12 is changed to 12. This keeps the data but prevents the "21" from pulling the model's math too far out of whack.

Option B: Keeping them as is: Modern models like XGBoost and Random Forest (which we planned to use) are actually very robust against outliers. They handle these "long tails" much better than simple Linear Regression.

## Data Cleaning & Encoding

uniqueid: This is a random string. It has no predictive power (e.g., being "User_123" doesn't make you more likely to have a bank account).

year: As noted by your Pandas Profiling alert, this is perfectly correlated with the country. It’s redundant data.



The Insight: This dataset was compiled from different FinScope surveys conducted in different years.

- Rwanda: 2016
- Tanzania: 2017
- Kenya & Uganda: 2018

What to do: Because every row for Rwanda says "2016" and every row for Tanzania says "2017," the year column doesn't actually add new information—it just repeats which country the data came from. You can safely drop the year column during preprocessing to simplify your model.

In [ ]:
df_cleaned = df.copy()
df_cleaned = df_cleaned.drop(columns=['uniqueid', 'year'])
print(df_cleaned.head())

In [ ]:
# Check unique values for the big categorical columns
cols_to_check = ['education_level', 'job_type', 'marital_status', 'country']
for col in cols_to_check:
    print(f"{col}: {df_cleaned[col].unique()}\n")

In [ ]:
df_cleaned.select_dtypes(include=['object']).columns

In [ ]:
# Map the target variable, bank_account
df_cleaned['bank_account'] = df_cleaned['bank_account'].map({'Yes': 1, 'No': 0})
df_cleaned.head()

In [ ]:
df_cleaned['location_type'] = df_cleaned['location_type'].map({'Rural': 0, 'Urban': 1})
df_cleaned['cellphone_access'] = df_cleaned['cellphone_access'].map({'No': 0, 'Yes': 1})
df_cleaned['gender_of_respondent'] = df_cleaned['gender_of_respondent'].map({'Female': 0, 'Male': 1})

df_cleaned.head()

In [ ]:
# Label Encoding Education Level
edu_map = {
    'No formal education': 0,
    'Other/Dont know/RTA': 0,
    'Primary education': 1,
    'Secondary education': 2,
    'Vocational/Specialised training': 3,
    'Tertiary education': 4
}
df_cleaned['education_level'] = df_cleaned['education_level'].map(edu_map)
df_cleaned.head()

In [ ]:
df_cleaned.select_dtypes(include=['object']).columns

In [ ]:
#OHE for the remaining variables
# This will turn the remaining 'Object' columns into multiple 1/0 columns
df_cleaned = pd.get_dummies(df_cleaned, columns=['job_type', 'marital_status', 'country', 'relationship_with_head'])
df_cleaned.head()

In [ ]:
df_cleaned.info()

In [ ]:
#Convert the boolean variables to integers
df_cleaned = df_cleaned.astype(int)
df_cleaned.head()

In [ ]:
df_cleaned.info()

Data Preprocessing & Feature Engineering

---

1. Data Cleaning (Dropping Columns)

To reduce noise and prevent the model from overfitting on non-predictive variables, the following columns were removed:

| Column | Reason for Removal |
|--------|--------------------|
| `uniqueid` | Unique respondent identifier with no predictive value for financial behavior. |
| `year` | Removed due to perfect correlation with the `country` column, which could introduce multicollinearity. |

---

2. Binary Encoding (Mapping)

Variables containing only two categories were converted into binary integers (`0` and `1`).

| Variable | Mapping |
|----------|---------|
| `bank_account` | No → 0, Yes → 1 |
| `location_type` | Rural → 0, Urban → 1 |
| `cellphone_access` | No → 0, Yes → 1 |
| `gender_of_respondent` | Female → 0, Male → 1 |

---

3. Ordinal Encoding (Hierarchy Mapping)

The `education_level` feature was manually encoded to preserve the natural hierarchy of educational attainment.

| Education Level | Encoded Value |
|-----------------|---------------|
| No formal education / Other / Don't know | 0 |
| Primary education | 1 |
| Secondary education | 2 |
| Vocational / Specialised training | 3 |
| Tertiary education | 4 |

---

4. One-Hot Encoding (Nominal Transformation)

For categorical variables without a natural order, **One-Hot Encoding** was applied using `pd.get_dummies()`.

| Variable | Transformation |
|----------|---------------|
| `country` | Created 4 binary columns (Kenya, Rwanda, Tanzania, Uganda) |
| `job_type` | Created 10 binary columns |
| `marital_status` | Created 5 binary columns |
| `relationship_with_head` | Created 6 binary columns |

This allowed the machine learning model to assign separate importance to each category.

---

5. Final Data Transformation

- Converted the final dataset to a uniform `int64` data type  
- Expanded the feature space from:
  - **13 original columns**
  - to **32 numerical features**

The dataset is now fully prepared for machine learning.

## Machine Learning - Predicting if one has a `bank_account`

### Logistic Regression

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, confusion_matrix, ConfusionMatrixDisplay

# Split data into X and y
y = df_cleaned['bank_account']
X = df_cleaned.drop('bank_account', axis=1)

# Split data into training and testing sets
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# Feature Scaling
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled = scaler.transform(X_val)

# Initialize and train the Logistic Regression model
model = LogisticRegression(random_state=42, solver='liblinear') # 'liblinear' is good for small datasets and binary classification
model.fit(X_train_scaled, y_train)

# Make predictions
y_pred = model.predict(X_val_scaled)
y_pred_proba = model.predict_proba(X_val_scaled)[:, 1]

# Print evaluation metrics
print("\n--- Model Evaluation ---")
print(f"Accuracy: {accuracy_score(y_val, y_pred):.4f}")
print(f"Precision: {precision_score(y_val, y_pred):.4f}")
print(f"Recall: {recall_score(y_val, y_pred):.4f}")
print(f"F1 Score: {f1_score(y_val, y_pred):.4f}")
print(f"ROC AUC Score: {roc_auc_score(y_val, y_pred_proba):.4f}")

# Plot Confusion Matrix
print("\n--- Confusion Matrix ---")
cm = confusion_matrix(y_val, y_pred)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=['No Bank Account', 'Has Bank Account'])
disp.plot(cmap=plt.cm.Blues)
plt.title('Confusion Matrix: Predicted vs. Actual')
plt.show()


#### Model Performance Interpretation

---

## 1. The Good News: Strong Discriminative Power

- **ROC AUC Score:** `0.8632`

This is a **strong result**, indicating that the model can effectively distinguish between:

- **Banked individuals**
- **Unbanked individuals**

### Interpretation
A ROC AUC above **0.85** suggests the model has:

- Good ranking capability  
- Strong separation between classes  
- Useful predictive potential  

The model understands the problem well at a probability level.

---

## 2. The Real Story: Precision vs Recall

### Precision: `0.7117`
When the model predicts that a person has a bank account, it is correct:

\[
71\%
\]

This means the model's positive predictions are reasonably reliable.

---

### Recall: `0.3505`
Out of all people who actually have bank accounts, the model only identifies:

\[
35\%
\]

This means the model misses:

\[
65\%
\]

of true banked individuals.

Recall is the main weakness of the model.

---

## 3. Confusion Matrix Breakdown

| Outcome | Count | Meaning |
|---------|-------|---------|
| True Negatives | 3949 | Correctly identified people without bank accounts |
| False Negatives | 430 | People with accounts missed by the model |
| True Positives | 232 | Correctly identified banked individuals |

---

### Key Insight

####  Strength
The model is very good at identifying:

- Individuals **without** bank accounts

####  Weakness
The model struggles to detect:

- Individuals who **do have** bank accounts

This creates a significant number of:

[430 False Negatives]

These are the most important errors because they represent financially included individuals the model fails to recognize.

---

## 4. Business Interpretation

Your model currently behaves conservatively:

- It avoids false alarms
- But misses many actual positive cases

This means the classifier favors:

 **Specificity**  
over  
 **Sensitivity**

---

## 5. Recommended Next Step

To improve performance, focus on increasing **Recall** by:

- Adjusting the classification threshold  
- Applying class balancing techniques (`SMOTE`)  
- Using weighted models (`class_weight='balanced'`)  
- Engineering stronger financial behavior features  

The goal is to detect more true banked individuals without sacrificing too much precision.

### Decision Tree Classifier

In [ ]:
from sklearn.tree import DecisionTreeClassifier

# Initialize and train the Decision Tree Classifier model
# Using the same random_state for reproducibility
dtc_model = DecisionTreeClassifier(max_depth=5, class_weight='balanced', random_state=42)
dtc_model.fit(X_train_scaled, y_train)

# Make predictions on the validation set
y_pred_dtc = dtc_model.predict(X_val_scaled)
y_pred_proba_dtc = dtc_model.predict_proba(X_val_scaled)[:, 1]

# Print evaluation metrics for Decision Tree Classifier
print("\n--- Decision Tree Classifier Model Evaluation ---")
print(f"Accuracy: {accuracy_score(y_val, y_pred_dtc):.4f}")
print(f"Precision: {precision_score(y_val, y_pred_dtc):.4f}")
print(f"Recall: {recall_score(y_val, y_pred_dtc):.4f}")
print(f"F1 Score: {f1_score(y_val, y_pred_dtc):.4f}")
print(f"ROC AUC Score: {roc_auc_score(y_val, y_pred_proba_dtc):.4f}")

# Plot Confusion Matrix for Decision Tree Classifier
print("\n--- Confusion Matrix (Decision Tree) ---")
cm_dtc = confusion_matrix(y_val, y_pred_dtc)
disp_dtc = ConfusionMatrixDisplay(confusion_matrix=cm_dtc, display_labels=['No Bank Account', 'Has Bank Account'])
disp_dtc.plot(cmap=plt.cm.Blues)
plt.title('Confusion Matrix: Predicted vs. Actual (Decision Tree)')
plt.show()

#### Decision Tree Model Performance Interpretation

---

## 1. The Big Win: High Recall

### Recall: `0.7402`

This is the strongest result of the model.

The model now correctly identifies:

\[
74\%
\]

of individuals who actually have bank accounts.

### Improvement from Baseline
Previous model recall:

\[
35\%
\]

Current model recall:

\[
74\%
\]

This represents a major improvement in identifying financially included individuals.

---

### Confusion Matrix Insight

Out of **662 actual account holders**:

- **490** were correctly identified (**True Positives**)  
- **172** were missed (**False Negatives**)  

This means the model is much better at detecting the target class.

---

## 2. The Trade-off: Lower Precision

### Precision: `0.3624`

When the model predicts that someone has a bank account, it is correct only:

\[
36\%
\]

of the time.

This indicates a large number of false positive predictions.

---

### False Positives: `862`

These are individuals the model predicted as banked but who actually are not.

### Business Meaning
If the bank used this model for outreach:

- Agents would contact many potential customers
- But a large portion would be incorrect leads

To find:

[490 true customers]

the bank would also contact:

[862 false leads]

---

## 3. Accuracy Decline

### Accuracy: `0.7802`

Overall accuracy dropped because the model is:

- Taking more risks  
- Predicting "Yes" more often  
- Prioritizing sensitivity over strict correctness  

This is expected when optimizing for recall.

---

## 4. ROC AUC Remains Strong

### ROC AUC: `0.8310`

Despite lower precision, the model still maintains a high ROC AUC score.

This means the model still has:

✅ Good class separation  
✅ Strong ranking ability  
✅ Useful predictive signal  

The model understands the difference between:

- Banked individuals  
- Unbanked individuals  

It is simply more aggressive in identifying positives.

---

## 5. Final Business Interpretation

The **Decision Tree model** is more suitable for:

- Financial inclusion campaigns  
- Customer acquisition  
- Expanding banking outreach  

Because in this context:

✅ Missing potential customers is worse than contacting too many.

---

## Summary

Compared to Logistic Regression:

- **Lower overall accuracy**
- **Lower precision**
- **Much higher recall**

### Key Insight
For a bank aiming to increase customer reach:

> It is better to over-identify potential customers  
> than to completely miss them.

### Random Forest

In [ ]:
from sklearn.ensemble import RandomForestClassifier

# Initialize and train the Random Forest Classifier model
# Using the same random_state for reproducibility and class_weight for imbalance
rf_model = RandomForestClassifier(n_estimators=100, max_depth=10, class_weight='balanced', random_state=42)
rf_model.fit(X_train_scaled, y_train)

# Make predictions on the validation set
y_pred_rf = rf_model.predict(X_val_scaled)
y_pred_proba_rf = rf_model.predict_proba(X_val_scaled)[:, 1]

# Print evaluation metrics for Random Forest Classifier
print("\n--- Random Forest Classifier Model Evaluation ---")
print(f"Accuracy: {accuracy_score(y_val, y_pred_rf):.4f}")
print(f"Precision: {precision_score(y_val, y_pred_rf):.4f}")
print(f"Recall: {recall_score(y_val, y_pred_rf):.4f}")
print(f"F1 Score: {f1_score(y_val, y_pred_rf):.4f}")
print(f"ROC AUC Score: {roc_auc_score(y_val, y_pred_proba_rf):.4f}")

# Plot Confusion Matrix for Random Forest Classifier
print("\n--- Confusion Matrix (Random Forest) ---")
cm_rf = confusion_matrix(y_val, y_pred_rf)
disp_rf = ConfusionMatrixDisplay(confusion_matrix=cm_rf, display_labels=['No Bank Account', 'Has Bank Account'])
disp_rf.plot(cmap=plt.cm.Blues)
plt.title('Confusion Matrix: Predicted vs. Actual (Random Forest)')
plt.show()

#### Random Forest Model Performance Interpretation

---

## 1. The Sweet Spot: Best F1 Score

### F1 Score: `0.5335`

This is the **highest F1 score** achieved so far.

### Comparison
| Model | F1 Score |
|-------|----------|
| Logistic Regression | 0.46 |
| Decision Tree | 0.48 |
| **Random Forest** | **0.5335** |

Because the **F1 Score balances Precision and Recall**, this indicates that the Random Forest provides the most stable overall performance.

✅ Best balance between:
- Correct positive predictions
- Capturing actual account holders

---

## 2. 📈 Strongest ROC AUC

### ROC AUC: `0.8712`

This is the highest AUC among all models.

An AUC of:

\[
87\%
\]

means the model has excellent ability to distinguish between:

- Individuals with bank accounts  
- Individuals without bank accounts  

### Interpretation
The model is highly reliable at ranking:

> Who is most likely to own a bank account.

---

## 3. Improved Precision

### Precision: `0.4151`

Precision improved compared to the Decision Tree.

### Comparison
| Model | Precision |
|-------|-----------|
| Decision Tree | 0.3624 |
| **Random Forest** | **0.4151** |

This means that when the model predicts a customer has a bank account, it is correct:

\[
41.5\%
\]

of the time.

---

## 4. Reduced False Positives

### False Positives Reduced

The confusion matrix shows:

| Model | False Positives |
|-------|----------------|
| Decision Tree | 862 |
| **Random Forest** | **696** |

By averaging predictions across many trees, the model became more selective and reduced unnecessary positive predictions.

### Interpretation
Using multiple trees helped the model become:

- More confident  
- Less noisy  
- More reliable  

---

## 5. Maintaining High Recall

### Recall: `0.7462`

The model still captures approximately:

\[
75\%
\]

of individuals who actually have bank accounts.

### Business Importance
This means the model remains highly effective for:

- Customer targeting  
- Financial inclusion programs  
- Banking outreach initiatives  

because it continues to identify most of the intended audience.

---

## 6. Final Interpretation

The Random Forest model delivers the best overall performance because it:

✅ Maintains high recall  
✅ Improves precision  
✅ Reduces false positives  
✅ Achieves the highest F1 score  
✅ Produces the strongest ROC AUC  

---

## 💡 Key Insight

Among all tested models, the **Random Forest** provides the best balance between:

- Finding true account holders  
- Avoiding unnecessary outreach  
- Maintaining predictive stability  

It represents the most practical model for real-world deployment.

### XGBoost

In [ ]:
from xgboost import XGBClassifier

# Initialize and train the XGBoost Classifier model
# Using the same random_state for reproducibility and scale_pos_weight for class imbalance
# scale_pos_weight is an important parameter for imbalanced datasets in XGBoost
# It's calculated as count(negative examples) / count(positive examples)
scale_pos_weight_value = (y_train == 0).sum() / (y_train == 1).sum()

xgb_model = XGBClassifier(objective='binary:logistic', eval_metric='logloss', use_label_encoder=False,
                          scale_pos_weight=scale_pos_weight_value, random_state=42)
xgb_model.fit(X_train_scaled, y_train)

# Make predictions on the validation set
y_pred_xgb = xgb_model.predict(X_val_scaled)
y_pred_proba_xgb = xgb_model.predict_proba(X_val_scaled)[:, 1]

# Print evaluation metrics for XGBoost Classifier
print("\n--- XGBoost Classifier Model Evaluation ---")
print(f"Accuracy: {accuracy_score(y_val, y_pred_xgb):.4f}")
print(f"Precision: {precision_score(y_val, y_pred_xgb):.4f}")
print(f"Recall: {recall_score(y_val, y_pred_xgb):.4f}")
print(f"F1 Score: {f1_score(y_val, y_pred_xgb):.4f}")
print(f"ROC AUC Score: {roc_auc_score(y_val, y_pred_proba_xgb):.4f}")

# Plot Confusion Matrix for XGBoost Classifier
print("\n--- Confusion Matrix (XGBoost) ---")
cm_xgb = confusion_matrix(y_val, y_pred_xgb)
disp_xgb = ConfusionMatrixDisplay(confusion_matrix=cm_xgb, display_labels=['No Bank Account', 'Has Bank Account'])
disp_xgb.plot(cmap=plt.cm.Blues)
plt.title('Confusion Matrix: Predicted vs. Actual (XGBoost)')
plt.show()

### Support Vector Machines

In [ ]:
from sklearn.svm import SVC

# Initialize and train the SVM model
# Using class_weight='balanced' to handle class imbalance
svm_model = SVC(kernel='linear', probability=True, class_weight='balanced', random_state=42)
svm_model.fit(X_train_scaled, y_train)

# Make predictions on the validation set
y_pred_svm = svm_model.predict(X_val_scaled)
y_pred_proba_svm = svm_model.predict_proba(X_val_scaled)[:, 1]

# Print evaluation metrics for SVM Classifier
print("\n--- Support Vector Machine Model Evaluation ---")
print(f"Accuracy: {accuracy_score(y_val, y_pred_svm):.4f}")
print(f"Precision: {precision_score(y_val, y_pred_svm):.4f}")
print(f"Recall: {recall_score(y_val, y_pred_svm):.4f}")
print(f"F1 Score: {f1_score(y_val, y_pred_svm):.4f}")
print(f"ROC AUC Score: {roc_auc_score(y_val, y_pred_proba_svm):.4f}")

# Plot Confusion Matrix for SVM Classifier
print("\n--- Confusion Matrix (SVM) ---")
cm_svm = confusion_matrix(y_val, y_pred_svm)
disp_svm = ConfusionMatrixDisplay(confusion_matrix=cm_svm, display_labels=['No Bank Account', 'Has Bank Account'])
disp_svm.plot(cmap=plt.cm.Blues)
plt.title('Confusion Matrix: Predicted vs. Actual (SVM)')
plt.show()

#### Support Vector Machine (SVM) Performance Interpretation

---

## 1. Strong Discriminative Power

### ROC AUC: `0.8636`

The SVM achieved a very strong ROC AUC score.

### Comparison
| Model | ROC AUC |
|-------|---------|
| Logistic Regression | ~0.863 |
| XGBoost | 0.8582 |
| **SVM** | **0.8636** |

This indicates that the selected demographic features maintain a stable relationship with banking status.

### Interpretation
SVM performs well because it can identify a consistent decision boundary between:

- Individuals with bank accounts  
- Individuals without bank accounts  

✅ The model is a strong classifier at the probability level.

---

## 2. High Recall, Lower Precision

### Recall: `0.7387`

The model correctly identified:

\[
74\%
\]

of all individuals who actually have bank accounts.

### Confusion Matrix Insight
Out of **662 actual account holders**:

- **489** correctly identified (**True Positives**)  
- **173** missed (**False Negatives**)  

This shows the model effectively captures the minority class.

---

### Precision: `0.3953`

When the model predicts a customer has a bank account, it is correct:

\[
39.5\%
\]

of the time.

This means the model still produces a large number of false alarms.

---

### False Positives

The model incorrectly predicted:

\[
748
\]

people as bank account holders when they were not.

### Interpretation
The SVM decision boundary appears slightly too broad, causing:

- Higher sensitivity  
- Lower precision  
- More noise in positive predictions  

---

## 3. Comparing the Top Models

Using **F1 Score** as the best measure of balanced performance:

| Model | F1 Score | Rank |
|-------|----------|------|
| Random Forest | 0.5335 | 🥇 |
| XGBoost | 0.5210 | 🥈 |
| SVM | 0.5150 | 🥉 |
| Decision Tree | 0.4866 | 4 |
| Logistic Regression | 0.4696 | 5 |

---

## 4. Key Insight

The SVM performed very well by:

✅ Maintaining high recall  
✅ Strong ROC AUC  
✅ Competitive overall performance  

However, it was slightly weaker than Random Forest because of:

Lower precision  
More false positives  

---

## 5. Final Conclusion

The SVM proved that the demographic variables contain a meaningful and predictable signal.

However, compared with the top-performing ensemble models:

**Random Forest remains the best overall model**

because it provides the strongest balance between:

- Precision  
- Recall  
- Stability  
- Business usefulness  

---

## Takeaway

> SVM confirms the dataset is highly predictable,  
> but ensemble models handle the complexity more effectively.

### Naive Bayes

In [ ]:
from sklearn.naive_bayes import GaussianNB

# Initialize and train the Naive Bayes model
nb_model = GaussianNB()
nb_model.fit(X_train_scaled, y_train)

# Make predictions on the validation set
y_pred_nb = nb_model.predict(X_val_scaled)
y_pred_proba_nb = nb_model.predict_proba(X_val_scaled)[:, 1]

# Print evaluation metrics for Naive Bayes Classifier
print("\n--- Naive Bayes Classifier Model Evaluation ---")
print(f"Accuracy: {accuracy_score(y_val, y_pred_nb):.4f}")
print(f"Precision: {precision_score(y_val, y_pred_nb):.4f}")
print(f"Recall: {recall_score(y_val, y_pred_nb):.4f}")
print(f"F1 Score: {f1_score(y_val, y_pred_nb):.4f}")
print(f"ROC AUC Score: {roc_auc_score(y_val, y_pred_proba_nb):.4f}")

# Plot Confusion Matrix for Naive Bayes Classifier
print("\n--- Confusion Matrix (Naive Bayes) ---")
cm_nb = confusion_matrix(y_val, y_pred_nb)
disp_nb = ConfusionMatrixDisplay(confusion_matrix=cm_nb, display_labels=['No Bank Account', 'Has Bank Account'])
disp_nb.plot(cmap=plt.cm.Blues)
plt.title('Confusion Matrix: Predicted vs. Actual (Naive Bayes)')
plt.show()

#### Naive Bayes Model Performance Interpretation

---

## 1. The Golden Mean: Balanced Performance

### Accuracy: `0.8527`

The Naive Bayes model achieved a strong overall accuracy.

This is notable because it remained close to the Logistic Regression baseline while offering more practical classification behavior.

---

### F1 Score: `0.5198`

The most impressive result is the F1 Score.

### Comparison
| Model | F1 Score |
|-------|----------|
| XGBoost | 0.5210 |
| **Naive Bayes** | **0.5198** |

For a relatively simple probabilistic model to perform nearly as well as XGBoost suggests:

✅ The selected features have strong independent predictive power.

---

## 2. Key Trade-Offs

### Recall: `0.5665`

The model correctly identified:

\[
56.7\%
\]

of individuals who actually have bank accounts.

### Confusion Matrix
Out of **662 actual account holders**:

- **375** correctly identified  
- **287** missed  

Compared to the tree-based models, recall is lower.

---

### Precision: `0.4802`

When the model predicts someone has a bank account, it is correct:

\[
48.0\%
\]

of the time.

This is actually higher than the Random Forest model.

### Comparison
| Model | Precision |
|-------|-----------|
| Random Forest | 0.4151 |
| **Naive Bayes** | **0.4802** |

---

## 3. Lower False Positives

### False Positives: `406`

Compared to Random Forest:

| Model | False Positives |
|-------|----------------|
| Random Forest | 696 |
| **Naive Bayes** | **406** |

This means Naive Bayes generated:

✅ Fewer false alarms  
✅ More conservative predictions  
✅ Less wasted outreach effort  

---

## 4. Confusion Matrix Interpretation

| Outcome | Count | Meaning |
|---------|-------|---------|
| True Negatives | 3637 | Correctly identified unbanked individuals |
| True Positives | 375 | Correctly identified banked individuals |
| False Positives | 406 | Incorrectly predicted as banked |

---

### Key Insight

Naive Bayes is very effective at:

- Identifying the unbanked  
- Limiting unnecessary positive predictions  
- Maintaining balanced performance  

---

## 5. Business Perspective

If the objective is:

### Expand reach aggressively
➡ Random Forest performs better because of higher recall.

### Minimize marketing waste
➡ Naive Bayes becomes a strong candidate because of higher precision and fewer false positives.

---

## 6. Final Interpretation

The Naive Bayes model stands out as:

✅ Simple  
✅ Efficient  
✅ Interpretable  
✅ Well-balanced  

It may not be the strongest model overall, but it offers one of the best trade-offs between:

- Precision  
- Accuracy  
- Practical deployment  

---

## Takeaway

> Naive Bayes proved that a simple model can remain highly competitive  
> when the underlying features carry meaningful signal.

### Final Model Comparison Leaderboard

| Model              | Accuracy | Recall | Precision | F1-Score | ROC AUC |
|-------------------|----------|--------|-----------|----------|---------|
| Random Forest      | 0.8164   | 0.7462 | 0.4151    | 0.5335   | 0.8712  |
| XGBoost            | 0.8108   | 0.7311 | 0.4047    | 0.5210   | 0.8582  |
| Naive Bayes        | 0.8527   | 0.5665 | 0.4802    | 0.5198   | 0.8332  |
| SVM                | 0.8043   | 0.7387 | 0.3953    | 0.5150   | 0.8636  |
| Logistic Regression| 0.8886   | 0.3505 | 0.7117    | 0.4696   | 0.8632  |

---

## 🎯 Key Takeaways

- **Best Overall Balance:** **Random Forest**
  - Highest **F1-Score**
  - Highest **ROC AUC**
  - Strongest balance between precision and recall  

- **Best Accuracy:** **Logistic Regression**
  - Highest raw accuracy
  - Weak recall makes it less useful for identifying account holders  

- **Best Precision:** **Logistic Regression**
  - Most conservative positive predictions  

- **Best Recall:** **Random Forest**
  - Captured the most actual bank account holders  

- **Best Lightweight Model:** **Naive Bayes**
  - Strong performance with simpler assumptions  

  Based on the comparative evaluation of all five models, **Random Forest is the strongest candidate for production deployment** because it delivers the best overall balance between predictive performance and business usefulness.
  
  While Logistic Regression achieved the highest accuracy, its low recall means it failed to identify many customers who actually have bank accounts, making it less effective for financial inclusion targeting.
  
  Random Forest achieved the **highest F1-score (0.5335)** and **highest ROC AUC (0.8712)**, demonstrating superior balance between correctly identifying banked individuals and maintaining reliable class separation.
  
  It also maintained the **highest recall (0.7462)**, which is critical in this project because missing potential customers carries a greater business cost than contacting extra prospects. Although its precision is lower than some simpler models, the model’s ability to consistently detect the target group makes it the most practical and strategic choice for real-world implementation.


  

#### Understanding Model Evaluation Metrics

---

## 1. Accuracy

- **Meaning:**  
  The percentage of total predictions the model got correct (both "Yes" and "No").

- **Higher is Better?**  
  Generally yes, but it can be misleading.  
  If 90% of people don’t have accounts, a model can achieve 90% accuracy by always predicting "No".

- **Best For:**  
  When classes are **balanced** (similar number of "Yes" and "No" cases).

---

## 2. Precision

- **Meaning:**  
  When the model predicts "Yes," how often is it actually correct?

- **Higher is Better?**  
  Yes. High precision means fewer false alarms.

- **Best For:**  
  Situations where **wrong positive predictions are costly**  
  (e.g., sending marketing resources unnecessarily).

---

## 3. Recall

- **Meaning:**  
  Out of all people who actually have an account, how many did the model correctly identify?

- **Higher is Better?**  
  Yes. High recall means fewer missed opportunities.

- **Best For:**  
  **Financial inclusion and outreach**, where missing true positives is costly.

---

## 4. F1-Score

- **Meaning:**  
  A balance between Precision and Recall.

- **Higher is Better?**  
  Yes. Prevents extreme models (too conservative or too aggressive).

- **Best For:**  
  **Imbalanced datasets**, where one class (like "Yes") is rare.

---

## 5. ROC AUC

- **Meaning:**  
  Measures how well the model distinguishes between the two classes.  
  If you randomly pick one "Yes" and one "No," how often does the model rank them correctly?

- **Higher is Better?**  
   Yes.  
  - **1.0** → Perfect model  
  - **0.5** → Random guessing  

- **Best For:**  
  Evaluating the model’s **overall intelligence**, regardless of threshold.

---

## Key Takeaway

> No single metric tells the full story.  
> The best model depends on your **business goal**:
>
> - Maximize **Recall** → Find more customers  
> - Maximize **Precision** → Reduce wasted effort  
> - Maximize **F1** → Balance both  

## Feature Importance

In [ ]:
# Get feature importance from your Random Forest Champion
importances = rf_model.feature_importances_
feature_names = X.columns
feature_importance_df = pd.DataFrame({'Feature': feature_names, 'Importance': importances})
feature_importance_df = feature_importance_df.sort_values(by='Importance', ascending=False)

# Plotting
plt.figure(figsize=(10, 8))
sns.barplot(x='Importance', y='Feature', data=feature_importance_df.head(10))
plt.title('Top 10 Most Important Features (Random Forest)')
plt.show()
plt.savefig('feature_importance_rf.png')

## Model Prediction on Test Data

In [ ]:
df_test = pd.read_csv('/content/Test.csv')

# Store uniqueid and country for submission
submission_id = df_test['uniqueid']
submission_country = df_test['country']

# Apply the same cleaning steps as training data
df_test_cleaned = df_test.copy()
df_test_cleaned = df_test_cleaned.drop(columns=['uniqueid', 'year'])

# Map binary columns
df_test_cleaned['location_type'] = df_test_cleaned['location_type'].map({'Rural': 0, 'Urban': 1})
df_test_cleaned['cellphone_access'] = df_test_cleaned['cellphone_access'].map({'No': 0, 'Yes': 1})
df_test_cleaned['gender_of_respondent'] = df_test_cleaned['gender_of_respondent'].map({'Female': 0, 'Male': 1})

# Label Encoding Education Level
edu_map = {
    'No formal education': 0,
    'Other/Dont know/RTA': 0,
    'Primary education': 1,
    'Secondary education': 2,
    'Vocational/Specialised training': 3,
    'Tertiary education': 4
}
df_test_cleaned['education_level'] = df_test_cleaned['education_level'].map(edu_map)

# One-Hot Encoding for remaining categorical variables
df_test_cleaned = pd.get_dummies(df_test_cleaned, columns=['job_type', 'marital_status', 'country', 'relationship_with_head'])

# Convert boolean columns to integers
df_test_cleaned = df_test_cleaned.astype(int)

# Align columns with X_train - crucial for consistent feature set
# Get missing columns in test set that are present in training set
missing_cols = set(X_train.columns) - set(df_test_cleaned.columns)
for c in missing_cols:
    df_test_cleaned[c] = 0
# Ensure the order of columns in the test set is the same as in the training set
df_test_processed = df_test_cleaned[X_train.columns]

# Scale the test features using the already fitted scaler
X_test_scaled = scaler.transform(df_test_processed)

# Make predictions using the Random Forest model
predictions_rf = rf_model.predict(X_test_scaled)

# Format for submission
submission_df = pd.DataFrame({
    'ID': submission_id + ' x ' + submission_country,
    'bank_account': predictions_rf
})

# Rename 'ID' column to 'uniqueid' as per Zindi submission requirements if needed, but it seems 'ID' is fine.
# Check Zindi submission format details if any specific column name is required for the ID.

# Export to CSV
submission_df.to_csv('submission.csv', index=False)

print("Submission file 'submission.csv' created successfully.")
print(submission_df.head())

In [ ]:
df_cleaned.to_csv('df_cleaned.csv', index=False)
print("df_cleaned exported to 'df_cleaned.csv'")